# Fake News Detection — Exploratory Data Analysis
Run after placing `True.csv` and `Fake.csv` in `data/raw/`

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import re

from src.data_prep import load_isot, build_combined_text

df = load_isot()
df = build_combined_text(df)
print(df.shape)
df.head()

In [ ]:
# Class distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df['label'].value_counts().plot(kind='bar', ax=axes[0], color=['#ef4444','#22c55e'])
axes[0].set_xticklabels(['Fake (0)', 'Real (1)'], rotation=0)
axes[0].set_title('Class Distribution')

# Article length distribution
df['text_len'] = df['combined_text'].str.split().str.len()
df.groupby('label')['text_len'].plot(kind='hist', bins=50, alpha=0.6, ax=axes[1], legend=True)
axes[1].set_title('Article Length Distribution (words)')
plt.tight_layout()
plt.show()

In [ ]:
# Top subjects
if 'subject' in df.columns:
    fig, ax = plt.subplots(figsize=(10, 4))
    df.groupby(['subject','label']).size().unstack().plot(kind='bar', ax=ax, color=['#ef4444','#22c55e'])
    ax.set_title('Articles per Subject by Class')
    plt.xticks(rotation=30, ha='right')
    plt.tight_layout()
    plt.show()

In [ ]:
# Most common words per class
from sklearn.feature_extraction.text import CountVectorizer

for label, name in [(0, 'Fake'), (1, 'Real')]:
    corpus = df[df.label == label]['combined_text'].tolist()
    vec = CountVectorizer(stop_words='english', max_features=15, ngram_range=(1,1))
    vec.fit(corpus)
    freqs = dict(zip(vec.get_feature_names_out(), vec.transform(corpus).toarray().sum(axis=0)))
    top = sorted(freqs.items(), key=lambda x: -x[1])[:15]
    words, counts = zip(*top)
    plt.figure(figsize=(8, 3))
    colour = '#ef4444' if name == 'Fake' else '#22c55e'
    plt.barh(list(words), list(counts), color=colour)
    plt.title(f'Top words in {name} articles')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()